# BME 2080 — Week 10
## Biological Interpretation & Model Evaluation

**How to navigate:**
- Everyone runs **Section 1** (setup — loads data and trains both models)
- EN teams (EVEN TEAMS) run **Section 2a**
- RF teams (ODD TEAMS) run **Section 2b**
- Everyone runs **Section 3** (mystery patients)
- If you finish early, feel free to try the **Optional Activity** in Section 4

---


## Section 1: Setup


In [ ]:
# Block 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

print('Libraries loaded successfully!')


In [ ]:
# Block 2: Load dataset
url = 'https://raw.githubusercontent.com/cr546-collab/bme2080-module3/main/teaching_multiomics_AUC_212lines.csv'
df = pd.read_csv(url)

feature_cols = ([c for c in df.columns if c.startswith('exp_')] +
                [c for c in df.columns if c.startswith('prot_')] +
                [c for c in df.columns if c.startswith('mut_')] +
                [c for c in df.columns if c.startswith('lineage_')])

drug_cols = sorted([c for c in df.columns if c.startswith('drug_')])

cancer_colors = {
    'Breast':                        '#e63946',
    'Colorectal':                    '#2a9d8f',
    'Lung Adenocarcinoma':           '#457b9d',
    'Lung Squamous Cell Carcinoma':  '#fb8500',
    'Ovary':                         '#8338ec',
    'Pancreas':                      '#06d6a0',
    'Skin':                          '#ffb703',
}

top4 = ['drug_Dabrafenib', 'drug_Vemurafenib', 'drug_Trametinib', 'drug_Afatinib']
top4_names = [d.replace('drug_', '') for d in top4]

# Feature type color mapping
FEAT_COLORS = {
    'mut_':     '#ff6b6b',  # red    — mutation status
    'exp_':     '#00bcd4',  # cyan   — RNA expression
    'prot_':    '#c77dff',  # purple — protein expression
    'lineage_': '#ffb703',  # gold   — cancer type / lineage
}

def feat_color(name, alpha=1.0):
    """Return RGBA color for a feature based on its type prefix."""
    import matplotlib.colors as mc
    for prefix, hex_col in FEAT_COLORS.items():
        if name.startswith(prefix):
            r, g, b = mc.to_rgb(hex_col)
            return (r, g, b, alpha)
    return (0.5, 0.5, 0.5, alpha)  # fallback gray

print(f'Dataset loaded: {df.shape[0]} cell lines, {len(feature_cols)} features, {len(drug_cols)} drugs')
print(f'Feature breakdown:')
print(f'  RNA expression (exp_):   {sum(1 for c in feature_cols if c.startswith("exp_"))} features')
print(f'  Protein expression (prot_): {sum(1 for c in feature_cols if c.startswith("prot_"))} features')
print(f'  Mutation status (mut_):  {sum(1 for c in feature_cols if c.startswith("mut_"))} features')
print(f'  Cancer type (lineage_):  {sum(1 for c in feature_cols if c.startswith("lineage_"))} features')


In [ ]:
# Block 3: Train EN and RF models on all 4 top drugs
# This uses the same settings as last week.

kf = KFold(n_splits=5, shuffle=True, random_state=42)
en_models, rf_models, scalers, imputers = {}, {}, {}, {}
dataset_means, dataset_stds = {}, {}

print('Training models...')
print(f'{"Drug":<15} {"EN alpha":>9} {"EN l1_ratio":>11} {"EN CV R²":>9} {"RF CV R²":>9}')
print('-' * 58)

for drug in top4:
    name = drug.replace('drug_', '')
    valid = df[df[drug].notna()].reset_index(drop=True)
    y = valid[drug].values
    dataset_means[name] = y.mean()
    dataset_stds[name]  = y.std()

    imp = SimpleImputer(strategy='mean')
    X   = imp.fit_transform(valid[feature_cols].values)
    sc  = StandardScaler()
    Xsc = sc.fit_transform(X)

    # Elastic Net — finds best alpha and l1_ratio automatically
    en = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
        alphas=np.logspace(-3, 2, 50),
        cv=5, max_iter=5000, random_state=42
    )
    en.fit(Xsc, y)

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=300, max_features='sqrt',
        min_samples_leaf=3, random_state=42, n_jobs=-1
    )
    rf.fit(X, y)

    # CV R² estimates
    en_scores, rf_scores = [], []
    for tr, te in kf.split(X):
        sc_cv = StandardScaler()
        en_cv = ElasticNetCV(
            l1_ratio=[0.1, 0.5, 0.9, 1.0],
            alphas=np.logspace(-3, 2, 20),
            cv=3, max_iter=2000, random_state=42
        )
        en_cv.fit(sc_cv.fit_transform(X[tr]), y[tr])
        en_scores.append(r2_score(y[te], en_cv.predict(sc_cv.transform(X[te]))))

        rf_cv = RandomForestRegressor(
            n_estimators=100, max_features='sqrt',
            min_samples_leaf=3, random_state=42, n_jobs=-1
        )
        rf_cv.fit(X[tr], y[tr])
        rf_scores.append(r2_score(y[te], rf_cv.predict(X[te])))

    en_models[name] = en
    rf_models[name] = rf
    scalers[name]   = sc
    imputers[name]  = imp

    print(f'{name:<15} {en.alpha_:>9.4f} {en.l1_ratio_:>11.2f} '
          f'{np.mean(en_scores):>9.3f} {np.mean(rf_scores):>9.3f}')

print('\nDone! Proceed to your assigned section.')


---
## Section 2a (EVEN): Elastic Net Coefficients

> **Only run this section if your team is assigned Elastic Net (EVEN TEAMS).**
> RF teams: jump to Section 2b.

---

### What are EN coefficients?

When Elastic Net trains, it learns a **coefficient** for every feature:

- **Bar extends left** (negative coefficient) → feature predicts **drug sensitivity** (lower AUC)
- **Bar extends right** (positive coefficient) → feature predicts **drug resistance** (higher AUC)
- **Feature not shown** → EN set its coefficient to zero and eliminated it

Bars are **colored by feature type** — mutation, RNA expression, protein expression,
or cancer type. Features predicting sensitivity are shown at full color; features
predicting resistance are shown faded.

**Key advantage over RF:** EN tells you both *what matters* AND *which direction*
it pushes the prediction. RF can only tell you *what matters*.

### What is the scatter plot showing?

Each dot is one cell line. The x-axis is the **actual measured AUC** from the
experiment. The y-axis is the **model's predicted AUC** using only molecular features.
A perfect model would place every dot exactly on the dashed diagonal.

The key question to ask: **do the predictions move with the actual values?**
If yes, the model learned something real. If predictions cluster near the middle
regardless of actual values, the model has little predictive power for individual cell lines.

Look for:
- Which drugs show dots clustering near the diagonal?
- Which show two distinct clusters vs. a continuous spread?
- How does the pattern in the scatter plot connect to what you see in the coefficient plot?


In [ ]:
# Block 4a: EN Coefficients + Predicted vs. Actual Scatter — all 4 top drugs
# Each row = one drug | Left = EN coefficients | Right = scatter plot
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

N_FEAT = 8
kf_scatter = KFold(n_splits=5, shuffle=True, random_state=42)

cancer_colors = {
    'Breast':                        '#e63946',
    'Colorectal':                    '#2a9d8f',
    'Lung Adenocarcinoma':           '#457b9d',
    'Lung Squamous Cell Carcinoma':  '#fb8500',
    'Ovary':                         '#8338ec',
    'Pancreas':                      '#06d6a0',
    'Skin':                          '#ffb703',
}

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
fig.patch.set_facecolor('white')
fig.suptitle('Elastic Net: Coefficients vs. Predicted vs. Actual — Top 4 Drugs\n'
             'Left: what the model learned  |  Right: how predictions look in the data',
             fontsize=13, fontweight='bold', color='#111111', y=1.01)

for row, (drug, name) in enumerate(zip(top4, top4_names)):
    valid  = df[df[drug].notna()].reset_index(drop=True)
    y      = valid[drug].values
    X      = imputers[name].transform(valid[feature_cols].values)

    # CV predictions for scatter
    preds  = np.zeros(len(y))
    scores = []
    for tr, te in kf_scatter.split(X):
        sc_cv = StandardScaler()
        from sklearn.linear_model import ElasticNetCV as ENCV
        en_cv = ENCV(l1_ratio=[0.1,0.5,0.9,1.0], alphas=np.logspace(-3,2,20),
                     cv=3, max_iter=2000, random_state=42)
        en_cv.fit(sc_cv.fit_transform(X[tr]), y[tr])
        preds[te] = en_cv.predict(sc_cv.transform(X[te]))
        scores.append(r2_score(y[te], preds[te]))
    cancers = valid['disease'].values

    # ── LEFT: coefficient plot ────────────────────────────────────────────────
    ax_c = axes[row][0]
    ax_c.set_facecolor('white')
    coefs = pd.Series(en_models[name].coef_, index=feature_cols)
    n_nz  = (coefs != 0).sum()
    top   = coefs.abs().nlargest(N_FEAT).index
    pc    = coefs[top].sort_values()
    colors = [feat_color(f, alpha=1.0 if v < 0 else 0.35)
              for f, v in zip(pc.index, pc.values)]
    ax_c.barh(range(len(pc)), pc.values, color=colors,
              edgecolor='#dddddd', linewidth=0.5)
    ax_c.set_yticks(range(len(pc)))
    ax_c.set_yticklabels(pc.index, fontsize=9, color='#222222')
    ax_c.axvline(0, color='#333333', lw=1.2, alpha=0.7)
    ax_c.set_title(f'EN Coefficients — {name}', fontsize=10,
                   fontweight='bold', color='#111111')
    ax_c.set_xlabel('Coefficient value', fontsize=8, color='#555555')
    ax_c.text(0.98, 0.02, f'{n_nz}/{len(feature_cols)} features used',
              transform=ax_c.transAxes, ha='right', va='bottom',
              fontsize=7.5, color='#888888', style='italic')
    ax_c.tick_params(colors='#444444', labelsize=8)
    for sp in ['top','right']: ax_c.spines[sp].set_visible(False)
    for sp in ['left','bottom']: ax_c.spines[sp].set_color('#cccccc')
    ax_c.grid(axis='x', alpha=0.3, color='#aaaaaa')

    # ── RIGHT: scatter plot ───────────────────────────────────────────────────
    ax_s = axes[row][1]
    ax_s.set_facecolor('white')
    for cancer in np.unique(cancers):
        idx = np.where(cancers == cancer)[0]
        ax_s.scatter(y[idx], preds[idx],
                     color=cancer_colors.get(cancer, '#999999'),
                     alpha=0.85, edgecolors='white', linewidths=0.4,
                     s=45, label=cancer)
    mn = min(y.min(), preds.min()) - 0.03
    mx = max(y.max(), preds.max()) + 0.03
    ax_s.plot([mn, mx], [mn, mx], 'k--', lw=1.5, alpha=0.4)
    rmse = np.sqrt(np.mean((y - preds)**2))
    ax_s.text(0.04, 0.96,
              f'CV R² = {np.mean(scores):.3f}\nRMSE = {rmse:.3f}\nn = {len(y)}',
              transform=ax_s.transAxes, fontsize=8.5, va='top',
              bbox=dict(boxstyle='round', facecolor='white',
                        edgecolor='#cccccc', alpha=0.9))
    ax_s.set_title(f'Predicted vs. Actual AUC — {name}', fontsize=10,
                   fontweight='bold', color='#111111')
    ax_s.set_xlabel('Actual AUC', fontsize=8, color='#555555')
    ax_s.set_ylabel('Predicted AUC', fontsize=8, color='#555555')
    ax_s.set_xlim(mn, mx); ax_s.set_ylim(mn, mx)
    ax_s.tick_params(colors='#444444', labelsize=8)
    for sp in ['top','right']: ax_s.spines[sp].set_visible(False)
    for sp in ['left','bottom']: ax_s.spines[sp].set_color('#cccccc')
    ax_s.grid(alpha=0.25, color='#aaaaaa')

# Legends
feat_handles = [
    mpatches.Patch(color='#ff6b6b', label='Mutation'),
    mpatches.Patch(color='#00bcd4', label='RNA expression'),
    mpatches.Patch(color='#c77dff', label='Protein expression'),
    mpatches.Patch(color='#ffb703', label='Cancer type / lineage'),
    mpatches.Patch(color='#aaaaaa', alpha=0.35, label='Faded = predicts resistance'),
]
cancer_handles = [mpatches.Patch(color=v, label=k)
                  for k, v in cancer_colors.items()]
fig.legend(handles=feat_handles, loc='lower left', fontsize=8,
           facecolor='white', edgecolor='#cccccc',
           title='Feature type (left panels)', title_fontsize=8,
           bbox_to_anchor=(0.01, -0.05), ncol=3)
fig.legend(handles=cancer_handles, loc='lower right', fontsize=8,
           facecolor='white', edgecolor='#cccccc',
           title='Cancer type (right panels)', title_fontsize=8,
           bbox_to_anchor=(0.99, -0.05), ncol=4)
plt.tight_layout()
plt.show()


---
## Section 2b (ODD): Random Forest Feature Importance

> **Only run this section if your team is assigned Random Forest (ODD TEAMS).**

---

### What is RF feature importance?

Random Forest measures **feature importance** as how much each feature reduced
prediction error across all trees and all splits in the forest.

- **Length** → a longer bar indicates that the feature was used frequently and contributed strongly
- **Direction** → importance scores are always positive; RF cannot tell you
  whether a feature predicts sensitivity or resistance

Bars are **colored by feature type**
so you can directly compare which types of features each model relies on.

**Why does RF weight mutations differently from EN?**
Mutation features are binary (0 or 1) and can only split the data once per branch.
Continuous expression features can be split at many different thresholds across
many levels of each tree, accumulating more importance. This doesn't mean mutations
are less biologically important; it's a consequence of how RF counts importance.

**Key limitation vs. EN:** RF cannot tell you *which direction* a feature pushes
the prediction. You can see what matters, but not whether it predicts sensitivity
or resistance.

### What is the scatter plot showing?

Each dot is one cell line. The x-axis is the **actual measured AUC** from the
experiment. The y-axis is the **model's predicted AUC** using only molecular features.
A perfect model would place every dot exactly on the dashed diagonal.

The key question to ask: **do the predictions move with the actual values?**
If yes, the model learned something real. If predictions cluster near the middle
regardless of actual values, the model has little predictive power for individual cell lines.

Look for:
- Which drugs show dots clustering near the diagonal?
- Which show two distinct clusters vs. a continuous spread?
- How does the pattern in the scatter plot connect to what you see in the coefficient plot?

In [ ]:
# Block 4b: RF Feature Importance + Predicted vs. Actual Scatter — all 4 top drugs
# Each row = one drug | Left = RF importance | Right = scatter plot
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

N_FEAT = 8
kf_scatter = KFold(n_splits=5, shuffle=True, random_state=42)

cancer_colors = {
    'Breast':                        '#e63946',
    'Colorectal':                    '#2a9d8f',
    'Lung Adenocarcinoma':           '#457b9d',
    'Lung Squamous Cell Carcinoma':  '#fb8500',
    'Ovary':                         '#8338ec',
    'Pancreas':                      '#06d6a0',
    'Skin':                          '#ffb703',
}

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
fig.patch.set_facecolor('white')
fig.suptitle('Random Forest: Feature Importance vs. Predicted vs. Actual — Top 4 Drugs\n'
             'Left: what the model learned  |  Right: how predictions look in the data',
             fontsize=13, fontweight='bold', color='#111111', y=1.01)

for row, (drug, name) in enumerate(zip(top4, top4_names)):
    valid  = df[df[drug].notna()].reset_index(drop=True)
    y      = valid[drug].values
    X      = imputers[name].transform(valid[feature_cols].values)

    # CV predictions for scatter
    preds  = np.zeros(len(y))
    scores = []
    for tr, te in kf_scatter.split(X):
        rf_cv = RandomForestRegressor(n_estimators=100, max_features='sqrt',
                                      min_samples_leaf=3, random_state=42, n_jobs=-1)
        rf_cv.fit(X[tr], y[tr])
        preds[te] = rf_cv.predict(X[te])
        scores.append(r2_score(y[te], preds[te]))
    cancers = valid['disease'].values

    # ── LEFT: importance plot ─────────────────────────────────────────────────
    ax_i = axes[row][0]
    ax_i.set_facecolor('white')
    importances = pd.Series(rf_models[name].feature_importances_, index=feature_cols)
    top    = importances.nlargest(N_FEAT).sort_values()
    colors = [feat_color(f, alpha=1.0) for f in top.index]
    ax_i.barh(range(len(top)), top.values, color=colors,
              edgecolor='#dddddd', linewidth=0.5)
    ax_i.set_yticks(range(len(top)))
    ax_i.set_yticklabels(top.index, fontsize=9, color='#222222')
    ax_i.set_title(f'RF Feature Importance — {name}', fontsize=10,
                   fontweight='bold', color='#111111')
    ax_i.set_xlabel('Importance score', fontsize=8, color='#555555')
    ax_i.tick_params(colors='#444444', labelsize=8)
    for sp in ['top','right']: ax_i.spines[sp].set_visible(False)
    for sp in ['left','bottom']: ax_i.spines[sp].set_color('#cccccc')
    ax_i.grid(axis='x', alpha=0.3, color='#aaaaaa')

    # ── RIGHT: scatter plot ───────────────────────────────────────────────────
    ax_s = axes[row][1]
    ax_s.set_facecolor('white')
    for cancer in np.unique(cancers):
        idx = np.where(cancers == cancer)[0]
        ax_s.scatter(y[idx], preds[idx],
                     color=cancer_colors.get(cancer, '#999999'),
                     alpha=0.85, edgecolors='white', linewidths=0.4,
                     s=45, label=cancer)
    mn = min(y.min(), preds.min()) - 0.03
    mx = max(y.max(), preds.max()) + 0.03
    ax_s.plot([mn, mx], [mn, mx], 'k--', lw=1.5, alpha=0.4)
    rmse = np.sqrt(np.mean((y - preds)**2))
    ax_s.text(0.04, 0.96,
              f'CV R² = {np.mean(scores):.3f}\nRMSE = {rmse:.3f}\nn = {len(y)}',
              transform=ax_s.transAxes, fontsize=8.5, va='top',
              bbox=dict(boxstyle='round', facecolor='white',
                        edgecolor='#cccccc', alpha=0.9))
    ax_s.set_title(f'Predicted vs. Actual AUC — {name}', fontsize=10,
                   fontweight='bold', color='#111111')
    ax_s.set_xlabel('Actual AUC', fontsize=8, color='#555555')
    ax_s.set_ylabel('Predicted AUC', fontsize=8, color='#555555')
    ax_s.set_xlim(mn, mx); ax_s.set_ylim(mn, mx)
    ax_s.tick_params(colors='#444444', labelsize=8)
    for sp in ['top','right']: ax_s.spines[sp].set_visible(False)
    for sp in ['left','bottom']: ax_s.spines[sp].set_color('#cccccc')
    ax_s.grid(alpha=0.25, color='#aaaaaa')

# Legends
feat_handles = [
    mpatches.Patch(color='#ff6b6b', label='Mutation'),
    mpatches.Patch(color='#00bcd4', label='RNA expression'),
    mpatches.Patch(color='#c77dff', label='Protein expression'),
    mpatches.Patch(color='#ffb703', label='Cancer type / lineage'),
]
cancer_handles = [mpatches.Patch(color=v, label=k)
                  for k, v in cancer_colors.items()]
fig.legend(handles=feat_handles, loc='lower left', fontsize=8,
           facecolor='white', edgecolor='#cccccc',
           title='Feature type (left panels)', title_fontsize=8,
           bbox_to_anchor=(0.01, -0.05), ncol=2)
fig.legend(handles=cancer_handles, loc='lower right', fontsize=8,
           facecolor='white', edgecolor='#cccccc',
           title='Cancer type (right panels)', title_fontsize=8,
           bbox_to_anchor=(0.99, -0.05), ncol=4)
plt.tight_layout()
plt.show()


---
## Section 3: Using the Models

> Run **Block 5a** and **Block 5b** in sequence.
> Discuss the questions at the end with your team.

---

### From features to predictions

You have now seen what EN and RF learned from the dataset. The next question is:
*what would these models recommend for a new patient we have never seen before?*

Below are 5 hypothetical patients with known molecular profiles. You will run each
patient through both EN and RF for all 4 top drugs — first looking at **which drug
is predicted to be most effective**, then asking **how confident the model is** in
those predictions.

| Patient | Cancer Type | Key Molecular Feature |
|---------|-------------|----------------------|
| **Patient A** | Skin (melanoma) | BRAF-mutated |
| **Patient B** | Colorectal | KRAS-mutated |
| **Patient C** | Lung adenocarcinoma | EGFR-mutated |
| **Patient D** | Skin (melanoma) | BRAF-mutated + NRAS co-mutation |
| **Patient E** | Ovary | No actionable mutations |


### Block 5a: Which drug is predicted to have the strongest effect?

AUC (Area Under the Curve) measures drug potency across a range of doses.
A **lower AUC** means the drug kills more cells — so the drug predicted to
have the lowest AUC is the model's strongest recommendation for that patient.

The dashed line on each plot shows the **dataset mean AUC** for each drug —
a useful reference for whether a prediction is meaningfully low or just average.

> **Important:** Different drugs have different baseline AUC distributions.
> A predicted AUC of 0.65 means something different for Dabrafenib (mean = 0.88)
> than for Trametinib (mean = 0.72). Keep this in mind when comparing across drugs.


In [ ]:
# Block 5a: Predicted AUC for all mystery patients
# Lower bar = stronger predicted drug effect
# Dashed line = dataset mean AUC for each drug

patient_url = 'https://raw.githubusercontent.com/cr546-collab/bme2080-module3/main/mystery_patients.csv'
patients_df = pd.read_csv(patient_url)

patient_labels = [
    'Patient A\nSkin | BRAF-mutated',
    'Patient B\nColorectal | KRAS-mutated',
    'Patient C\nLung Adeno | EGFR-mutated',
    'Patient D\nSkin | BRAF+NRAS',
    'Patient E\nOvary | No mutations',
]

fig, axes = plt.subplots(1, 5, figsize=(26, 6), sharey=True)
fig.patch.set_facecolor('white')

x     = np.arange(len(top4_names))
width = 0.32

# Store predictions for Block 5b
all_en_preds, all_rf_preds = [], []

for ax, (_, row), label in zip(axes, patients_df.iterrows(), patient_labels):
    ax.set_facecolor('white')
    patient = row[feature_cols].values.reshape(1, -1)

    en_preds, rf_preds, means = [], [], []
    for name in top4_names:
        p_imp = imputers[name].transform(patient)
        en_p  = en_models[name].predict(scalers[name].transform(p_imp))[0]
        rf_p  = rf_models[name].predict(p_imp)[0]
        en_preds.append(en_p)
        rf_preds.append(rf_p)
        means.append(dataset_means[name])

    all_en_preds.append(en_preds)
    all_rf_preds.append(rf_preds)

    bars_en = ax.bar(x - width/2, en_preds, width, label='Elastic Net',
                     color='#ff9f43', alpha=0.9, edgecolor='#cccccc', linewidth=0.5)
    bars_rf = ax.bar(x + width/2, rf_preds, width, label='Random Forest',
                     color='#00bcd4', alpha=0.9, edgecolor='#cccccc', linewidth=0.5)

    for i, m in enumerate(means):
        ax.plot([x[i]-width, x[i]+width], [m, m],
                color='#888888', lw=1.8, ls='--', alpha=0.8)

    for bar in bars_en:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7, color='#cc6600')
    for bar in bars_rf:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7, color='#007a8a')

    ax.set_xticks(x)
    ax.set_xticklabels(top4_names, rotation=30, ha='right', fontsize=9, color='#222222')
    ax.set_title(label, fontsize=9, fontweight='bold', color='#111111', pad=8)
    ax.set_ylim(0, 1.18)
    ax.tick_params(colors='#333333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    ax.grid(axis='y', alpha=0.3, color='#aaaaaa')

axes[0].set_ylabel('Predicted AUC\n(lower = stronger predicted drug effect)',
                    fontsize=10, color='#333333')

import matplotlib.patches as mpatches
en_patch  = mpatches.Patch(color='#ff9f43', label='Elastic Net prediction')
rf_patch  = mpatches.Patch(color='#00bcd4', label='Random Forest prediction')
mean_line = plt.Line2D([0],[0], color='#888888', ls='--', lw=2,
                        label='Dataset mean AUC')
fig.legend(handles=[en_patch, rf_patch, mean_line],
           loc='lower center', ncol=3, fontsize=10,
           facecolor='white', edgecolor='#cccccc',
           bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Block 5a: Predicted Drug Effect — Which Drug Has the Lowest Predicted AUC?\n'
             '(lower bar = stronger predicted effect; dashed line = dataset mean)',
             fontsize=12, fontweight='bold', color='#111111', y=1.02)
plt.tight_layout()
plt.show()

print('Block 5a complete. Proceed to Block 5b to evaluate prediction confidence.')


### Block 5b: How confident are we in those predictions?

Block 5a showed which drug has the lowest predicted AUC, but a low absolute AUC
isn't the whole story. Some drugs naturally have lower average AUC across all cell
lines (e.g. Trametinib mean = 0.72) while others are higher (Dabrafenib mean = 0.88).
A prediction of 0.65 means something very different for each drug.

Block 5b normalizes for this by showing how far each prediction falls below the
**dataset mean for that drug**, measured in standard deviations:

- **Higher bar** → model is making a stronger, more unusual prediction of sensitivity
- **Bar near zero** → model predicts average response — not a strong recommendation
- **Bar below zero** → model predicts this patient is more resistant than average

Use both plots together:
1. **Block 5a** tells you which drug would have the strongest absolute effect
2. **Block 5b** tells you how much the model has actually "learned" something
   specific about this patient, vs. just predicting near the average

> When Block 5a and 5b agree on the top drug, you can be more confident in the
> recommendation. When they disagree, ask why: is the model predicting a meaningful
> effect, or is it just picking the drug with the lowest baseline?


In [ ]:
# Block 5b: Prediction confidence — how far below the dataset mean?
# Uses predictions already computed in Block 5a (must run Block 5a first)
# Higher bar = stronger, more unusual prediction of sensitivity

fig, axes = plt.subplots(1, 5, figsize=(26, 6), sharey=True)
fig.patch.set_facecolor('white')

x     = np.arange(len(top4_names))
width = 0.32

pnames = ['Patient A', 'Patient B', 'Patient C', 'Patient D', 'Patient E']
all_en_sds, all_rf_sds = [], []

for ax, label, en_preds, rf_preds in zip(axes, patient_labels, all_en_preds, all_rf_preds):
    ax.set_facecolor('white')

    en_sds = [(dataset_means[n] - ep) / dataset_stds[n]
              for n, ep in zip(top4_names, en_preds)]
    rf_sds = [(dataset_means[n] - rp) / dataset_stds[n]
              for n, rp in zip(top4_names, rf_preds)]
    all_en_sds.append(en_sds)
    all_rf_sds.append(rf_sds)

    bars_en = ax.bar(x - width/2, en_sds, width, label='Elastic Net',
                     color='#ff9f43', alpha=0.9, edgecolor='#cccccc', linewidth=0.5)
    bars_rf = ax.bar(x + width/2, rf_sds, width, label='Random Forest',
                     color='#00bcd4', alpha=0.9, edgecolor='#cccccc', linewidth=0.5)

    for bar in bars_en:
        h = bar.get_height()
        va = 'bottom' if h >= 0 else 'top'
        ax.text(bar.get_x()+bar.get_width()/2, h + (0.04 if h>=0 else -0.04),
                f'{h:+.2f}', ha='center', va=va, fontsize=7, color='#cc6600')
    for bar in bars_rf:
        h = bar.get_height()
        va = 'bottom' if h >= 0 else 'top'
        ax.text(bar.get_x()+bar.get_width()/2, h + (0.04 if h>=0 else -0.04),
                f'{h:+.2f}', ha='center', va=va, fontsize=7, color='#007a8a')

    ax.axhline(0,   color='#333333', lw=1.2, alpha=0.7)
    ax.axhline(0.5, color='#aaaaaa', lw=1,   ls=':',  alpha=0.6)
    ax.axhline(1.0, color='#888888', lw=1,   ls='--', alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(top4_names, rotation=30, ha='right', fontsize=9, color='#222222')
    ax.set_title(label, fontsize=9, fontweight='bold', color='#111111', pad=8)
    ax.tick_params(colors='#333333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    ax.grid(axis='y', alpha=0.2, color='#aaaaaa')

axes[0].set_ylabel('Sensitivity signal\n(SDs below dataset mean — higher = stronger signal)',
                    fontsize=10, color='#333333')

en_patch  = mpatches.Patch(color='#ff9f43', label='Elastic Net')
rf_patch  = mpatches.Patch(color='#00bcd4', label='Random Forest')
line_1    = plt.Line2D([0],[0], color='#888888', ls='--', lw=1.5,
                        label='1 SD below mean (strong signal)')
line_half = plt.Line2D([0],[0], color='#aaaaaa', ls=':', lw=1.5,
                        label='0.5 SD below mean (moderate signal)')
fig.legend(handles=[en_patch, rf_patch, line_1, line_half],
           loc='lower center', ncol=4, fontsize=9,
           facecolor='white', edgecolor='#cccccc',
           bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Block 5b: Prediction Confidence — How Far Below Average Is Each Prediction?\n'
             '(higher bar = model is making a stronger, more unusual prediction of sensitivity)',
             fontsize=12, fontweight='bold', color='#111111', y=1.02)
plt.tight_layout()
plt.show()

# Agreement summary
print('\nModel recommendation summary:')
print(f'{"Patient":<12} {"5a: lowest AUC (EN)":>20} {"5a: lowest AUC (RF)":>20} '
      f'{"5b: strongest signal (EN)":>26} {"5b: strongest signal (RF)":>26}')
print('-' * 108)
for pname, en_preds, rf_preds, en_sds, rf_sds in zip(
        pnames, all_en_preds, all_rf_preds, all_en_sds, all_rf_sds):
    en_raw  = top4_names[en_preds.index(min(en_preds))]
    rf_raw  = top4_names[rf_preds.index(min(rf_preds))]
    en_sd   = top4_names[en_sds.index(max(en_sds))]
    rf_sd   = top4_names[rf_sds.index(max(rf_sds))]
    print(f'{pname:<12} {en_raw:>20} {rf_raw:>20} {en_sd:>26} {rf_sd:>26}')


---
## Section 4: Optional Activity (if you're bored)

> **Feel free to explore if you have finished Sections 1–3.**

---

### Feature Sparsity: How Many Features Does Each Drug Model Actually Use?

Elastic Net eliminates irrelevant features by setting their coefficients to zero.
The number of features it keeps varies dramatically across drugs, and that
difference is itself a biological signal.

Run the cell below, and consider the following questions.

**Reflection Questions:**
1. Which drug uses the fewest features? Which uses the most?
2. What does a high number of features tell you about the biological complexity
   of predicting that drug's response vs. a low number?
3. EN using only 9/143 features for Dabrafenib might sound like the model
   is throwing information away. Is that actually a problem, or is it a feature?
4. If you were presenting these results to a clinician, which drug model would
   you feel most confident recommending, and why?


In [ ]:
# Extension: Feature sparsity comparison
sparsity = {name: (pd.Series(en_models[name].coef_, index=feature_cols) != 0).sum()
            for name in top4_names}

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.bar(sparsity.keys(), sparsity.values(),
              color='#ff9f43', edgecolor='#cccccc', linewidth=0.5, alpha=0.9)
ax.axhline(len(feature_cols), color='#888888', lw=1.5, ls='--',
           alpha=0.7, label=f'Total available features: {len(feature_cols)}')
for bar, val in zip(bars, sparsity.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, str(val),
            ha='center', va='bottom', color='#333333',
            fontsize=12, fontweight='bold')

ax.set_ylabel('Features used (non-zero EN coefficients)', color='#333333', fontsize=11)
ax.set_title('EN Feature Sparsity: How Many Features Does Each Drug Model Use?',
             color='#111111', fontsize=12, fontweight='bold')
ax.tick_params(colors='#333333')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.3, color='#aaaaaa')
ax.legend(fontsize=10, facecolor='white', edgecolor='#cccccc')
ax.set_ylim(0, len(feature_cols) + 15)
plt.tight_layout()
plt.show()

print('EN feature sparsity summary:')
for name, n in sparsity.items():
    pct = n / len(feature_cols) * 100
    print(f'  {name:<15}: {n:>3} features used ({pct:.1f}% of {len(feature_cols)} total)')
